<a href="https://colab.research.google.com/github/Surya-Teja-PS/LLM-Evaluation-Framework/blob/main/LLM_Evaluation_Framework.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
!pip install -U -q groq google-genai deepeval pandas matplotlib

In [10]:
# ============================================================
# LLM Evaluation Framework — Study Assistant
# Extends the original single-example notebook into a real
# evaluation pipeline: a test dataset, three scoring methods,
# aggregation, and cross-method agreement analysis.
# ============================================================

# ---- 1. Setup ------------------------------------------------
# !pip install -U -q groq google-genai deepeval pandas matplotlib

from google.colab import userdata
from groq import Groq
from google import genai
from google.genai.errors import ClientError
import pandas as pd
import matplotlib.pyplot as plt
import re
import json
import time

groq_client = Groq(api_key=userdata.get("GROQ_API_KEY"))
gemini_client = genai.Client(api_key=userdata.get("GEMINI_API_KEY"))

# Gemini free tier caps gemini-3.6-flash at ~20 requests/day. This
# pipeline makes 3 Gemini calls per test case (1 manual judge + 2
# DeepEval metrics), so lower TEST_QUESTIONS below or run across
# multiple days if you're on the free tier. This wrapper auto-retries
# on 429s using the server's suggested retryDelay instead of crashing.

def call_with_retry(fn, *args, max_retries=5, **kwargs):
    for attempt in range(max_retries):
        try:
            return fn(*args, **kwargs)
        except ClientError as e:
            if e.code == 429:
                # Try to read the server's suggested wait time; default to 60s
                wait_s = 60
                try:
                    for detail in e.details.get("error", {}).get("details", []):
                        if "retryDelay" in detail:
                            wait_s = int(float(detail["retryDelay"].rstrip("s"))) + 2
                except Exception:
                    pass
                print(f"  Rate limited. Waiting {wait_s}s before retry ({attempt+1}/{max_retries})...")
                time.sleep(wait_s)
            else:
                raise
    raise RuntimeError("Max retries exceeded on Gemini rate limit.")

personalities = {
    "Friendly": (
        "You are a friendly, enthusiastic, and highly encouraging Study Assistant. "
        "Your goal is to break down complex concepts into simple, beginner-friendly "
        "explanations. Use analogies and real-world examples that beginners can relate "
        "to. Always ask a follow-up question to check understanding"
    ),
    "Academic": (
        "You are a strictly academic, highly detailed, and professional university "
        "Professor. Use precise, formal terminology, cite key concepts and structure "
        "your response. Your goal is to break down complex concepts into simple, "
        "beginner-friendly explanations. Use analogies and real-world examples that "
        "beginners can relate to. Always ask a follow-up question to check understanding"
    ),
}


def study_assistant(question: str, persona: str) -> str:
    system_prompt = personalities[persona]
    response = groq_client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": question},
        ],
    )
    return response.choices[0].message.content


# ---- 2. Test dataset ------------------------------------------
# 10 questions x 2 personas = 20 test cases. Spans easy/medium/hard
# so the eval isn't just testing one difficulty level.

TEST_QUESTIONS = [
    {"question": "What are LLMs?", "difficulty": "easy"},
    {"question": "What is the difference between supervised and unsupervised learning?", "difficulty": "easy"},
    {"question": "How does gradient descent work?", "difficulty": "medium"},
    {"question": "What is overfitting and how do you prevent it?", "difficulty": "medium"},
    {"question": "Explain the attention mechanism in transformers.", "difficulty": "hard"},
    {"question": "What is backpropagation?", "difficulty": "medium"},
    {"question": "Why do neural networks need activation functions?", "difficulty": "medium"},
    {"question": "What is the difference between precision and recall?", "difficulty": "easy"},
    {"question": "How does reinforcement learning from human feedback (RLHF) work?", "difficulty": "hard"},
    {"question": "What is the vanishing gradient problem?", "difficulty": "hard"},
]

TEST_CASES = [
    {"question": q["question"], "difficulty": q["difficulty"], "persona": persona}
    for q in TEST_QUESTIONS
    for persona in personalities.keys()
]  # -> 20 test cases (3 Gemini calls each = 60 calls; free tier ~20/day)
TEST_CASES = TEST_CASES[:6]
# On the Gemini free tier, slice this down to fit your daily quota, e.g.:
# TEST_CASES = TEST_CASES[:6]  # ~6 cases = ~18 Gemini calls, safely under 20/day


# ---- 3. Manual LLM-as-judge -----------------------------------

EVAL_PROMPT = """You are an expert evaluator for AI assistants.

Evaluate the Study Assistant's response based on these 6 criteria.
Score each from 1-5 (5 = excellent, 3 = acceptable, 1 = poor).

## Evaluation Criteria:

1. **Accuracy** (1-5): Is the information factually correct?
2. **Clarity** (1-5): Is it easy for a beginner (12th class student) to understand?
3. **Relevance** (1-5): Does it directly answer the question asked?
4. **Use of Analogies** (1-5): Does it use real-world examples or analogies?
5. **Follow-up Question** (1-5): Does it include a question to check understanding?
6. **Persona Consistency** (1-5): Does the tone match the expected persona?
   - "Friendly": enthusiastic, encouraging, warm
   - "Academic": formal, precise, professional

## Input Details:

**Student Question:** {question}
**Expected Persona:** {persona}
**Assistant Response:** {response}

## Your Evaluation:

Respond in EXACTLY this format (numbers only, no extra text):

ACCURACY: [score]
CLARITY: [score]
RELEVANCE: [score]
ANALOGIES: [score]
FOLLOW_UP: [score]
PERSONA: [score]
"""


def evaluate_response(question: str, persona: str, response: str) -> dict:
    prompt = EVAL_PROMPT.format(question=question, persona=persona, response=response)
    result = call_with_retry(
        gemini_client.models.generate_content,
        model="gemini-3.6-flash", contents=prompt
    )
    text = result.text

    scores = {}
    for criterion in ["ACCURACY", "CLARITY", "RELEVANCE", "ANALOGIES", "FOLLOW_UP", "PERSONA"]:
        match = re.search(rf"{criterion}:\s*(\d)", text)
        scores[criterion.lower()] = int(match.group(1)) if match else None
    return scores


# ---- 4. DeepEval metrics ---------------------------------------
# !pip install -q deepeval

from deepeval.models import GeminiModel
from deepeval.metrics import AnswerRelevancyMetric, GEval
from deepeval.test_case import LLMTestCase, LLMTestCaseParams

gemini_judge = GeminiModel(model="gemini-3.6-flash", api_key=userdata.get("GEMINI_API_KEY"))

relevance_metric = AnswerRelevancyMetric(threshold=0.7, model=gemini_judge, include_reason=True)

accuracy_metric = GEval(
    name="Accuracy",
    criteria=(
        "Determine if the response contains factually correct information about the "
        "topic. The explanation should be accurate and free from errors. Students "
        "should not learn incorrect concepts."
    ),
    evaluation_params=[LLMTestCaseParams.INPUT, LLMTestCaseParams.ACTUAL_OUTPUT],
    model=gemini_judge,
    threshold=0.7,
)


def run_deepeval(question: str, response: str) -> dict:
    test_case = LLMTestCase(input=question, actual_output=response)

    call_with_retry(relevance_metric.measure, test_case)
    call_with_retry(accuracy_metric.measure, test_case)

    return {
        "deepeval_relevancy_score": relevance_metric.score,
        "deepeval_relevancy_pass": relevance_metric.is_successful(),
        "deepeval_accuracy_score": accuracy_metric.score,
        "deepeval_accuracy_pass": accuracy_metric.is_successful(),
    }


# ---- 5. Run the full pipeline across the test set ---------------

def run_evaluation_suite(test_cases: list) -> pd.DataFrame:
    rows = []
    for i, case in enumerate(test_cases):
        question = case["question"]
        persona = case["persona"]
        difficulty = case["difficulty"]

        print(f"[{i+1}/{len(test_cases)}] {persona} — {question[:50]}...")

        response = study_assistant(question, persona)
        manual_scores = evaluate_response(question, persona, response)
        deepeval_scores = run_deepeval(question, response)
        time.sleep(2)  # small pacing gap between test cases

        row = {
            "question": question,
            "persona": persona,
            "difficulty": difficulty,
            "response": response,
            **manual_scores,
            **deepeval_scores,
        }
        rows.append(row)

    return pd.DataFrame(rows)


# ---- 6. Aggregation + reporting ----------------------------------

def summarize(df: pd.DataFrame):
    manual_cols = ["accuracy", "clarity", "relevance", "analogies", "follow_up", "persona"]

    print("\n=== Average manual-judge scores (out of 5) ===")
    print(df[manual_cols].mean().round(2))

    print("\n=== Average manual-judge scores by persona ===")
    print(df.groupby("persona")[manual_cols].mean().round(2))

    print("\n=== Average manual-judge scores by difficulty ===")
    print(df.groupby("difficulty")[manual_cols].mean().round(2))

    print("\n=== DeepEval pass rates ===")
    print("Relevancy pass rate:", df["deepeval_relevancy_pass"].mean())
    print("Accuracy pass rate:", df["deepeval_accuracy_pass"].mean())

    # Cross-method agreement: does the manual judge's accuracy score (>=4 = "pass")
    # agree with DeepEval's GEval accuracy pass/fail?
    df["manual_accuracy_pass"] = df["accuracy"] >= 4
    agreement = (df["manual_accuracy_pass"] == df["deepeval_accuracy_pass"]).mean()
    print(f"\n=== Manual judge vs GEval accuracy agreement: {agreement:.0%} ===")

    return agreement


def plot_persona_comparison(df: pd.DataFrame, out_path="persona_comparison.png"):
    manual_cols = ["accuracy", "clarity", "relevance", "analogies", "follow_up", "persona"]
    grouped = df.groupby("persona")[manual_cols].mean()

    ax = grouped.T.plot(kind="bar", figsize=(9, 5))
    ax.set_ylabel("Average score (out of 5)")
    ax.set_title("Study Assistant: Friendly vs Academic persona, by criterion")
    ax.set_ylim(0, 5)
    plt.xticks(rotation=30, ha="right")
    plt.tight_layout()
    plt.savefig(out_path, dpi=150)
    plt.show()


# ---- 7. Main --------------------------------------------------

if __name__ == "__main__":
    results_df = run_evaluation_suite(TEST_CASES)
    results_df.to_csv("eval_results.csv", index=False)

    agreement_rate = summarize(results_df)
    plot_persona_comparison(results_df)

    print("\nSaved raw results to eval_results.csv")
    print("Saved chart to persona_comparison.png")

/tmp/ipykernel_2689/481979330.py:163: DeprecationWarning: 'LLMTestCaseParams' is deprecated and will be removed in a future release. Use 'SingleTurnParams' instead.
  from deepeval.test_case import LLMTestCase, LLMTestCaseParams


[1/6] Friendly — What are LLMs?...
  Rate limited. Waiting 35s before retry (1/5)...
  Rate limited. Waiting 60s before retry (2/5)...
  Rate limited. Waiting 60s before retry (3/5)...
  Rate limited. Waiting 60s before retry (4/5)...
  Rate limited. Waiting 60s before retry (5/5)...


RuntimeError: Max retries exceeded on Gemini rate limit.